# Technical Note P3: NumPy Indexing, Slicing, Broadcasting

**Wook Kang · Revised English edition · Version 1.0.0 · 3 October 2026**

This note connects a mathematical task to a numerical object, an implementation, and evidence about the result. It is an instructional computational document. Demonstration parameters are illustrative unless explicitly stated otherwise.

Run from a fresh kernel in cell order. See [the series README](../README.md) for setup and [editorial and verification notes](../docs/EDITORIAL_NOTES.md) for scope and limitations. The English prose has been reorganized and expanded from the supplied Korean notebook; this is an adapted edition rather than a line-by-line translation.

**Reading question:** What is given, what is unknown, what operation relates them, and what independent check can assess the output?

## Core mathematical structure

$$
\boxed{
\text{shape}
\quad\text{and}\quad
\text{axis}
}
$$

$$
u_{i-1},\quad u_i,\quad u_{i+1}
$$

$$
\boxed{
\text{indexing}
\rightarrow
\text{local value}
}
$$

$$
\boxed{
\text{slicing}
\rightarrow
\text{region / neighbor relation}
}
$$

$$
\boxed{
\text{broadcasting}
\rightarrow
\text{shape-compatible field algebra}
}
$$

In [1]:
# Reproducible display and scale-aware comparison.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
def scaled_allclose(a, b, rtol=1e-12, atol=None, **kwargs):
    """Compare numerical examples using an absolute tolerance scaled to b."""
    a, b = np.asarray(a), np.asarray(b)
    scale = float(np.max(np.abs(b))) if b.size else 0.0
    if atol is None:
        atol = 1e-14 * max(scale, np.finfo(float).tiny)
    return np.allclose(a, b, rtol=rtol, atol=atol, **kwargs)


## 1. Indexing and slicing

An integer index selects an entry and can remove an axis. A slice selects a range while retaining the corresponding axis.

```python
u = np.array([10, 20, 30, 40, 50])
```

```python
u[2]
```

```python
u[1:4]
```

In [2]:
import numpy as np

u = np.array([10, 20, 30, 40, 50])

print("u[2]   =", u[2])
print("u[1:4] =", u[1:4])

u[2]   = 30
u[1:4] = [20 30 40]


## 2. Zero-based indexing

The first entry has index zero; negative indices count from the end. Translate physical point numbering into this convention explicitly.

```python
u[0]
```

```python
u[-1]
```

In [3]:
u = np.array([10, 20, 30, 40, 50])

print("first =", u[0])
print("last  =", u[-1])

first = 10
last  = 50


$$
u_1,u_2,\ldots,u_N
$$

```python
u[0], u[1], ..., u[N-1]
```

## 3. Slice syntax

A slice is start:stop:step, with stop excluded. Its length and location must match the intended boundary, interior, or neighbour set.

```python
a[start:stop:step]
```

In [4]:
a = np.arange(10)

print("a        =", a)
print("a[2:7]   =", a[2:7])
print("a[:5]    =", a[:5])
print("a[5:]    =", a[5:])
print("a[::2]   =", a[::2])
print("a[::-1]  =", a[::-1])

a        = [0 1 2 3 4 5 6 7 8 9]
a[2:7]   = [2 3 4 5 6]
a[:5]    = [0 1 2 3 4]
a[5:]    = [5 6 7 8 9]
a[::2]   = [0 2 4 6 8]
a[::-1]  = [9 8 7 6 5 4 3 2 1 0]


```python
a[2:7]
```

$$
2,3,4,5,6
$$

## 4. One-dimensional boundaries and interior

Use endpoint indices for nodal boundaries and 1:-1 for the interior. A cell-centred boundary lies at a face rather than at the first cell centre.

```python
u.shape == (N,)
```

In [5]:
u = np.arange(8, dtype=float)

print("u =", u)
print("left boundary  =", u[0])
print("right boundary =", u[-1])
print("interior       =", u[1:-1])

u = [0. 1. 2. 3. 4. 5. 6. 7.]
left boundary  = 0.0
right boundary = 7.0
interior       = [1. 2. 3. 4. 5. 6.]


$$
\boxed{
\texttt{u[1:-1]}
\leftrightarrow
\text{all interior points}
}
$$

## 5. Neighbour slices

Aligned shifted slices represent neighbouring samples. Their shapes must agree before differences can be computed elementwise.

$$
\frac{d^2u}{dx^2}\Big|_i
\approx
\frac{u_{i+1}-2u_i+u_{i-1}}{\Delta x^2}
$$

```python
u[:-2]
u[1:-1]
u[2:]
```

In [6]:
u = np.array([0., 1., 4., 9., 16., 25.])

left   = u[:-2]
center = u[1:-1]
right  = u[2:]

print("left   =", left)
print("center =", center)
print("right  =", right)

left   = [0. 1. 4. 9.]
center = [ 1.  4.  9. 16.]
right  = [ 4.  9. 16. 25.]


| NumPy slice | Mathematical meaning |  
| --- | --- |  
| `u[:-2]` | $u_{i-1}$ |  
| `u[1:-1]` | $u_i$ |  
| `u[2:]` | $u_{i+1}$ |

In [7]:
dx = 1.0
d2u = (u[2:] - 2*u[1:-1] + u[:-2]) / dx**2

print("second difference =", d2u)

second difference = [2. 2. 2. 2.]


```python
for i in range(1, N-1):
    ...
```

## 6. Loops and slicing

Vectorization changes how independent updates are evaluated. If a loop intentionally updates in place and uses newly updated neighbours, a vectorized simultaneous update is a different algorithm.

In [8]:
u = np.arange(10, dtype=float)**2
dx = 1.0

# Loop
d2_loop = np.zeros_like(u)
for i in range(1, len(u)-1):
    d2_loop[i] = (u[i+1] - 2*u[i] + u[i-1]) / dx**2

# Slicing
d2_slice = np.zeros_like(u)
d2_slice[1:-1] = (u[2:] - 2*u[1:-1] + u[:-2]) / dx**2

print("loop   =", d2_loop)
print("slice  =", d2_slice)
print("equal? =", scaled_allclose(d2_loop, d2_slice))

loop   = [0. 2. 2. 2. 2. 2. 2. 2. 2. 0.]
slice  = [0. 2. 2. 2. 2. 2. 2. 2. 2. 0.]
equal? = True


## 7. Two-dimensional indexing

Under the ij convention, the first index is x and the second is y. Keep this convention consistent in boundary masks and plots.

```python
U.shape == (Nx,Ny)
```

```python
U[i,j]
```

In [9]:
U = np.arange(20).reshape(4, 5)

print(U)
print("U[2,3] =", U[2, 3])

[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
U[2,3] = 13


```python
U[i,j]
```

$$
u(x_i,y_j)
$$

## 8. Two-dimensional boundaries

Each boundary fixes one index and varies the other. Corners belong to two boundaries and require an explicit convention when conditions differ.

```python
U[0, :]   # x-
U[-1, :]  # x+
U[:, 0]   # y-
U[:, -1]  # y+
```

In [10]:
U = np.arange(20).reshape(4, 5)

print("x- boundary:", U[0, :])
print("x+ boundary:", U[-1, :])
print("y- boundary:", U[:, 0])
print("y+ boundary:", U[:, -1])

x- boundary: [0 1 2 3 4]
x+ boundary: [15 16 17 18 19]
y- boundary: [ 0  5 10 15]
y+ boundary: [ 4  9 14 19]


$$
u=0
$$

In [11]:
U = np.ones((4, 5))

U[0, :] = 0.0

print(U)

[[0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]


$$
\boxed{
\text{boundary condition}
\rightarrow
\text{boundary slice assignment}
}
$$

## 9. Two-dimensional interior

The slice 1:-1,1:-1 excludes the outer rows and columns. It represents an interior set, not a complete boundary-value operator.

```python
U[1:-1, 1:-1]
```

In [12]:
U = np.arange(30).reshape(5, 6)

print("full:")
print(U)

print("\ninterior:")
print(U[1:-1, 1:-1])

full:
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]
 [24 25 26 27 28 29]]

interior:
[[ 7  8  9 10]
 [13 14 15 16]
 [19 20 21 22]]


```python
(Nx-2, Ny-2)
```

## 10. Two-dimensional neighbours

Shift one coordinate slice while keeping the others aligned. This is the array form of a local stencil.

$$
\nabla^2u=
\frac{u_{i+1,j}-2u_{i,j}+u_{i-1,j}}{\Delta x^2}
+
\frac{u_{i,j+1}-2u_{i,j}+u_{i,j-1}}{\Delta y^2}
$$

In [13]:
Nx, Ny = 6, 7
dx, dy = 1.0, 1.0

x = np.arange(Nx)
y = np.arange(Ny)
X, Y = np.meshgrid(x, y, indexing="ij")

U = X**2 + Y**2

lap = np.zeros_like(U, dtype=float)

lap[1:-1, 1:-1] = (
    (U[2:, 1:-1] - 2*U[1:-1, 1:-1] + U[:-2, 1:-1]) / dx**2
    +
    (U[1:-1, 2:] - 2*U[1:-1, 1:-1] + U[1:-1, :-2]) / dy**2
)

print("interior Laplacian:")
print(lap[1:-1, 1:-1])

interior Laplacian:
[[4. 4. 4. 4. 4.]
 [4. 4. 4. 4. 4.]
 [4. 4. 4. 4. 4.]
 [4. 4. 4. 4. 4.]]


```python
(Nx-2, Ny-2)
```

## 11. Advanced indexing

Integer index arrays select specified entries. Advanced indexing returns a copy; repeated indices require care during scatter accumulation.

```python
a[[0,2,5]]
```

In [14]:
a = np.arange(10)

idx = [0, 2, 5, 8]
print(a[idx])

[0 2 5 8]


## 12. Boolean indexing

A boolean mask selects entries satisfying a condition. Check that its shape and physical location correspond to the field being filtered.

$$
u>0.5
$$

In [15]:
u = np.linspace(0, 1, 11)

mask = u > 0.5

print("u =", u)
print("mask =", mask)
print("selected =", u[mask])

u = [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
mask = [False False False False False False  True  True  True  True  True]
selected = [0.6 0.7 0.8 0.9 1. ]


## 13. Modify a masked region

Mask assignment updates the original array. Preserve an independent copy when the original state is needed for a later comparison.

$$
u<0
$$

In [16]:
u = np.array([-1.0, -0.2, 0.1, 0.5])

u[u < 0] = 0.0

print(u)

[0.  0.  0.1 0.5]


$$
\boxed{
\text{condition}
\rightarrow
\text{mask}
\rightarrow
\text{selective field operation}
}
$$

## 14. Where

Where chooses between already evaluated array expressions. It does not generally protect an invalid expression in an unselected branch from being evaluated.

$$
u(x)=
\begin{cases}
u_L, & x<x_c\\
u_R, & x\ge x_c
\end{cases}
$$

In [17]:
x = np.linspace(0, 1, 11)

u = np.where(x < 0.5, 0.3, 0.1)

print(u)

[0.3 0.3 0.3 0.3 0.3 0.1 0.1 0.1 0.1 0.1 0.1]


## 15. Slices are usually views

A basic slice can share storage with its parent. Editing the slice can therefore change the original field.

```python
b = a[1:4]
```

In [18]:
a = np.array([0, 1, 2, 3, 4, 5])
b = a[1:4]

b[0] = 100

print("a =", a)
print("b =", b)

a = [  0 100   2   3   4   5]
b = [100   2   3]


$$
\boxed{
\text{slicing}
\approx
\text{shared-memory view}
}
$$

## 16. Independent copies

Copy explicitly when a new state must be independent. This is essential when comparing old and new time levels.

```python
b = a[1:4].copy()
```

In [19]:
a = np.array([0, 1, 2, 3, 4, 5])
b = a[1:4].copy()

b[0] = 100

print("a =", a)
print("b =", b)

a = [0 1 2 3 4 5]
b = [100   2   3]


```python
u_old = u
```

```python
u_old = u.copy()
```

## 17. Check memory sharing

Shares_memory tests whether arrays overlap in storage. Shape equality does not establish either independence or aliasing.

In [20]:
a = np.arange(10)
b = a[2:7]
c = a[2:7].copy()

print("a & b share memory?", np.shares_memory(a, b))
print("a & c share memory?", np.shares_memory(a, c))

a & b share memory? True
a & c share memory? False


## 18. Broadcasting

Broadcasting aligns trailing axes and expands singleton axes conceptually. The result's physical meaning still requires an axis convention.

```python
A.shape == (3,4)
b.shape == (4,)
```

```python
A + b
```

In [21]:
A = np.arange(12).reshape(3, 4)
b = np.array([10, 20, 30, 40])

print("A shape =", A.shape)
print("b shape =", b.shape)
print(A + b)

A shape = (3, 4)
b shape = (4,)
[[10 21 32 43]
 [14 25 36 47]
 [18 29 40 51]]


## 19. Broadcasting rules

Aligned lengths must match or one must equal one. Missing leading axes are treated as singleton axes.

```python
(3,4)
(   4)
```

```python
(3,1)
(1,4)
```

```python
(3,4)
```

In [22]:
a = np.array([[1], [2], [3]])      # (3,1)
b = np.array([[10, 20, 30, 40]])   # (1,4)

C = a + b

print("a shape =", a.shape)
print("b shape =", b.shape)
print("C shape =", C.shape)
print(C)

a shape = (3, 1)
b shape = (1, 4)
C shape = (3, 4)
[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


## 20. A field without dense meshgrid arrays

Coordinate columns and rows can broadcast to form a sampled field. This can reduce coordinate storage while retaining the full resulting field.

$$
(N_x,1)
$$

$$
(1,N_y)
$$

In [23]:
x = np.linspace(0, 1, 4)[:, None]   # (Nx,1)
y = np.linspace(0, 2, 5)[None, :]   # (1,Ny)

U = x + 2*y

print("x shape =", x.shape)
print("y shape =", y.shape)
print("U shape =", U.shape)
print(U)

x shape = (4, 1)
y shape = (1, 5)
U shape = (4, 5)
[[0.         1.         2.         3.         4.        ]
 [0.33333333 1.33333333 2.33333333 3.33333333 4.33333333]
 [0.66666667 1.66666667 2.66666667 3.66666667 4.66666667]
 [1.         2.         3.         4.         5.        ]]


$$
u(x,y)=x+2y
$$

$$
U_{ij}=x_i+2y_j
$$

## 21. Property fields

A scalar coefficient, a spatial coefficient field, and a componentwise coefficient vector require different shapes. Insert singleton axes to state the intended dependence.

```python
Dx.shape == (Nx,1)
U.shape  == (Nx,Ny)
```

```python
Dx * U
```

In [24]:
Nx, Ny = 4, 5

Dx = np.linspace(1.0, 2.0, Nx)[:, None]
U = np.ones((Nx, Ny))

Q = Dx * U

print("Dx shape =", Dx.shape)
print("U shape  =", U.shape)
print("Q shape  =", Q.shape)
print(Q)

Dx shape = (4, 1)
U shape  = (4, 5)
Q shape  = (4, 5)
[[1.         1.         1.         1.         1.        ]
 [1.33333333 1.33333333 1.33333333 1.33333333 1.33333333]
 [1.66666667 1.66666667 1.66666667 1.66666667 1.66666667]
 [2.         2.         2.         2.         2.        ]]


## 22. A failed broadcast

Read the two shapes from the trailing axis backward. An exception is useful when the requested elementwise relationship is undefined.

In [25]:
A = np.zeros((3, 4))
b = np.zeros(5)

try:
    C = A + b
except ValueError as e:
    print("Expected broadcasting error:")
    print(e)

Expected broadcasting error:
operands could not be broadcast together with shapes (3,4) (5,) 


## 23. Prepare singleton axes

None and newaxis make the intended expansion visible. Do not assume that equal axis lengths imply equal physical meanings.

```python
x.shape == (Nx,)
```

```python
x[:,None]
```

```python
(Nx,1)
```

```python
y[None,:]
```

```python
(1,Ny)
```

In [26]:
x = np.arange(4)
y = np.arange(5)

print("x      =", x.shape)
print("x[:,None] =", x[:, None].shape)
print("y      =", y.shape)
print("y[None,:] =", y[None, :].shape)

x      = (4,)
x[:,None] = (4, 1)
y      = (5,)
y[None,:] = (1, 5)


## 24. Three-dimensional coordinates

Use one varying coordinate axis and singleton lengths for the others. Sparse coordinate storage does not eliminate the memory needed for the resulting full field.

```python
x[:,None,None]   -> (Nx,1,1)
y[None,:,None]   -> (1,Ny,1)
z[None,None,:]   -> (1,1,Nz)
```

In [27]:
x = np.linspace(0, 1, 4)
y = np.linspace(0, 1, 3)
z = np.linspace(0, 1, 2)

X = x[:, None, None]
Y = y[None, :, None]
Z = z[None, None, :]

U = X + Y + Z

print("X shape =", X.shape)
print("Y shape =", Y.shape)
print("Z shape =", Z.shape)
print("U shape =", U.shape)

X shape = (4, 1, 1)
Y shape = (1, 3, 1)
Z shape = (1, 1, 2)
U shape = (4, 3, 2)


## 25. Elementwise and matrix products

The star operator multiplies entries; the at operator contracts matrix axes. They represent different mathematical operations even when both expressions execute.

```python
A * B
```

```python
A @ B
```

In [28]:
A = np.array([[1., 2.],
              [3., 4.]])
B = np.array([[10., 20.],
              [30., 40.]])

print("A * B =")
print(A * B)

print("\nA @ B =")
print(A @ B)

A * B =
[[ 10.  40.]
 [ 90. 160.]]

A @ B =
[[ 70. 100.]
 [150. 220.]]


## 26. Scalar broadcasting

A scalar acts on every entry. Its units and meaning must be appropriate for all components or samples involved.

```python
U + 1.0
2.0 * U
```

In [29]:
U = np.arange(6).reshape(2, 3)

print(U + 10)
print(0.5 * U)

[[10 11 12]
 [13 14 15]]
[[0.  0.5 1. ]
 [1.5 2.  2.5]]


$$
u_{ij}+c
$$

## 27. Boundary-profile broadcasting

A boundary profile belongs to the boundary's varying coordinate. Insert the axis needed to apply it to the intended rows or columns.

$$
u(0,y)=g(y)
$$

```python
U[0,:]
```

In [30]:
Nx, Ny = 5, 6
U = np.zeros((Nx, Ny))

y = np.linspace(0, 1, Ny)
g = 0.2 + 0.1*np.sin(np.pi*y)

U[0, :] = g

print(U)

[[0.2        0.25877853 0.29510565 0.29510565 0.25877853 0.2       ]
 [0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.        ]]


## 28. A sliced first derivative

A centred derivative aligns left and right neighbours at each interior point. Endpoint treatment is a separate choice.

$$
\frac{du}{dx}\Big|_i
\approx
\frac{u_{i+1}-u_{i-1}}{2\Delta x}
$$

In [31]:
x = np.linspace(0, 1, 11)
dx = x[1] - x[0]
u = x**2

dudx = np.zeros_like(u)
dudx[1:-1] = (u[2:] - u[:-2]) / (2*dx)

print("x       =", x)
print("numeric =", dudx)
print("exact   =", 2*x)

x       = [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
numeric = [0.  0.2 0.4 0.6 0.8 1.  1.2 1.4 1.6 1.8 0. ]
exact   = [0.  0.2 0.4 0.6 0.8 1.  1.2 1.4 1.6 1.8 2. ]


$$
u_{i+1}
\leftrightarrow
\texttt{u[2:]}
$$

$$
u_{i-1}
\leftrightarrow
\texttt{u[:-2]}
$$

## 29. Face differences

Adjacent cell values define a gradient between centres. The face array has a different location and length from the cell array.

$$
u_i
$$

$$
\left.\frac{du}{dx}\right|_{i+1/2}
\approx
\frac{u_{i+1}-u_i}{\Delta x}
$$

In [32]:
u = np.array([0.30, 0.28, 0.24, 0.20, 0.17])
dx = 0.01

grad_face = (u[1:] - u[:-1]) / dx

print("u[:-1] =", u[:-1])
print("u[1:]  =", u[1:])
print("face gradients =", grad_face)

u[:-1] = [0.3  0.28 0.24 0.2 ]
u[1:]  = [0.28 0.24 0.2  0.17]
face gradients = [-2. -4. -4. -3.]


```python
u[:-1]
u[1:]
```

## 30. Face diffusivity

Assign a coefficient at the same faces as the gradient. Harmonic averaging represents two equal-distance scalar resistances in series under specific assumptions.

$$
D_{i+1/2}=
\frac{D_i+D_{i+1}}{2}
$$

In [33]:
D = np.array([1.0, 1.2, 1.5, 2.0, 2.5])

D_face = 0.5*(D[:-1] + D[1:])

print("D_face =", D_face)

D_face = [1.1  1.35 1.75 2.25]


$$
q_{i+1/2}=
-D_{i+1/2}
\frac{u_{i+1}-u_i}{\Delta x}
$$

In [34]:
u = np.array([0.30, 0.28, 0.24, 0.20, 0.17])
D = np.array([1.0, 1.2, 1.5, 2.0, 2.5])
dx = 0.01

D_face = 0.5*(D[:-1] + D[1:])
grad_face = (u[1:] - u[:-1]) / dx
q_face = -D_face * grad_face

print("q_face =", q_face)

q_face = [2.2  5.4  7.   6.75]


```python
(N-1,)
```

## 31. Neighbours in the x direction

Shift the axis declared as x. A familiar-looking stencil can be wrong if its storage axis convention differs.

```python
U[:-1, :]
U[1:, :]
```

In [35]:
U = np.arange(20).reshape(4, 5)

print("left cells shape  =", U[:-1, :].shape)
print("right cells shape =", U[1:, :].shape)

left cells shape  = (3, 5)
right cells shape = (3, 5)


$$
u_{i+1,j}-u_{i,j}
$$

```python
U[1:,:] - U[:-1,:]
```

In [36]:
dUx_face = U[1:, :] - U[:-1, :]
print(dUx_face)

[[5 5 5 5 5]
 [5 5 5 5 5]
 [5 5 5 5 5]]


## 32. Neighbours in the y direction

Shift the axis declared as y. Use the corresponding spacing rather than assuming equal grid widths.

```python
U[:, :-1]
U[:, 1:]
```

In [37]:
dUy_face = U[:, 1:] - U[:, :-1]

print("shape =", dUy_face.shape)
print(dUy_face)

shape = (4, 4)
[[1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 1]]


## 33. Cell and face shapes

Internal faces number one fewer than cells in each direction. Complete face arrays additionally include both boundary faces.

```python
U.shape == (Nx,Ny)
```

```python
qx.shape == (Nx-1,Ny)
```

```python
qy.shape == (Nx,Ny-1)
```

$$
\boxed{
\text{cell-centered field}
\neq
\text{face-centered field}
}
$$

In [38]:
Nx, Ny = 6, 4
U = np.zeros((Nx, Ny))

qx = U[1:, :] - U[:-1, :]
qy = U[:, 1:] - U[:, :-1]

print("U  shape =", U.shape)
print("qx shape =", qx.shape)
print("qy shape =", qy.shape)

U  shape = (6, 4)
qx shape = (5, 4)
qy shape = (6, 3)


## 34. Divergence from face fluxes

Subtract outgoing and incoming face fluxes with consistent signs and divide by cell width or volume. Shared face fluxes enable telescoping conservation.

$$
\frac{q_{i+1/2}-q_{i-1/2}}{\Delta x}
$$

In [39]:
q_face = np.array([1.0, 0.7, 0.4, 0.2, 0.1])
dx = 0.01

div_q = (q_face[1:] - q_face[:-1]) / dx

print("div_q =", div_q)

div_q = [-30. -30. -20. -10.]


$$
\boxed{
\text{neighbor difference}
\rightarrow
\text{gradient}
}
$$

$$
\boxed{
\text{face-flux difference}
\rightarrow
\text{divergence}
}
$$

## 35. Masks as geometry

A mask identifies a region but does not automatically supply boundary fluxes or geometric accuracy. Stair-step boundaries may require a different discretization.

In [40]:
x = np.linspace(-1, 1, 101)
y = np.linspace(-1, 1, 101)
X, Y = np.meshgrid(x, y, indexing="ij")

mask = X**2 + Y**2 <= 0.6**2

U = np.zeros_like(X)
U[mask] = 1.0

print("active cells =", np.count_nonzero(mask))
print("total cells  =", mask.size)

active cells = 2815
total cells  = 10201


## 36. Cartesian index selection

Ix_ selects the Cartesian product of index sets. Paired advanced indices instead select corresponding pairs.

In [41]:
A = np.arange(25).reshape(5, 5)

rows = [0, 2, 4]
cols = [1, 3]

sub = A[np.ix_(rows, cols)]

print(A)
print("\nselected block:")
print(sub)

[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]
 [20 21 22 23 24]]

selected block:
[[ 1  3]
 [11 13]
 [21 23]]


## 37. Ellipsis

Ellipsis fills unspecified axes. It is useful when component axes are known but the number of leading spatial or batch axes varies.

```python
V[...,0]
```

In [42]:
V = np.zeros((4, 5, 6, 3))

vx = V[..., 0]

print("V shape  =", V.shape)
print("vx shape =", vx.shape)

V shape  = (4, 5, 6, 3)
vx shape = (4, 5, 6)


```python
V[...,0]
```

```python
V[:,:,:,0]
```

## 38. Select tensor components

Select component axes independently of location axes. A single component is generally basis dependent rather than an invariant property.

```python
sigma.shape == (Nx,Ny,Nz,3,3)
```

```python
sigma[...,0,0]
```

$$
\sigma_{xx}
$$

In [43]:
sigma = np.zeros((4, 5, 6, 3, 3))

sigma_xx = sigma[..., 0, 0]
sigma_xy = sigma[..., 0, 1]

print("sigma_xx shape =", sigma_xx.shape)
print("sigma_xy shape =", sigma_xy.shape)

sigma_xx shape = (4, 5, 6)
sigma_xy shape = (4, 5, 6)


## 39. Scale a vector field

A spatial scalar field usually needs a final singleton axis to multiply all components at each point. Verify the resulting shape.

```python
a.shape == (Nx,Ny,Nz)
```

```python
V.shape == (Nx,Ny,Nz,3)
```

In [44]:
a = np.ones((4, 5, 6))
V = np.ones((4, 5, 6, 3))

W = a[..., None] * V

print("a shape       =", a.shape)
print("a[...,None]   =", a[..., None].shape)
print("V shape       =", V.shape)
print("W shape       =", W.shape)

a shape       = (4, 5, 6)
a[...,None]   = (4, 5, 6, 1)
V shape       = (4, 5, 6, 3)
W shape       = (4, 5, 6, 3)


$$
a(\mathbf x)\mathbf v(\mathbf x)
$$

```python
a[...,None] * V
```

## 40. Diagonal anisotropic transport

Componentwise multiplication is valid when the diffusivity tensor is diagonal in the chosen basis. Off-diagonal terms require a contraction.

$$
\mathbf D=
\operatorname{diag}(D_x,D_y,D_z)
$$

```python
gx, gy, gz
```

$$
q_x=-D_xg_x,\quad
q_y=-D_yg_y,\quad
q_z=-D_zg_z
$$

In [45]:
shape = (4, 5, 6)

gx = np.ones(shape)
gy = 2*np.ones(shape)
gz = 3*np.ones(shape)

Dx = 1.0*np.ones(shape)
Dy = 0.5*np.ones(shape)
Dz = 2.0*np.ones(shape)

qx = -Dx*gx
qy = -Dy*gy
qz = -Dz*gz

print(qx.shape, qy.shape, qz.shape)

(4, 5, 6) (4, 5, 6) (4, 5, 6)


## 41. Broadcasted assignment

Assignment can broadcast values into a target slice. Confirm the intended repeated values before using this convenience for boundary conditions.

```python
U[:,0] = 0.5
```

In [46]:
U = np.zeros((4, 5))
U[:, 0] = 0.5

print(U)

[[0.5 0.  0.  0.  0. ]
 [0.5 0.  0.  0.  0. ]
 [0.5 0.  0.  0.  0. ]
 [0.5 0.  0.  0.  0. ]]


## 42. In-place arithmetic

In-place updates modify aliases and must fit the target dtype. A state update that overwrites old data can also change the numerical method.

```python
U *= 2
```

```python
V = U * 2
```

In [47]:
U = np.array([1.0, 2.0, 3.0])
V = U.copy()

U *= 2
W = V * 2

print("U =", U)
print("V =", V)
print("W =", W)

U = [2. 4. 6.]
V = [1. 2. 3.]
W = [2. 4. 6.]


## 43. Clip

Clipping enforces bounds algebraically but can alter mass or other invariants. It is not a substitute for an admissible conservative update.

$$
0\le u\le1
$$

In [48]:
u = np.array([-0.2, 0.1, 0.5, 1.2])

u_clipped = np.clip(u, 0.0, 1.0)

print(u_clipped)

[0.  0.1 0.5 1. ]


## 44. Finite-value masks

Check isfinite to identify NaN and infinity. Intentional NaN boundary sentinels should remain distinguishable from a failed interior calculation.

In [49]:
u = np.array([1.0, np.nan, 2.0, np.inf])

print("isfinite =", np.isfinite(u))
print("invalid locations =", np.where(~np.isfinite(u))[0])

isfinite = [ True False  True False]
invalid locations = [1 3]


```python
np.all(np.isfinite(u))
```

## 45. Reason about slice shapes

Calculate the number and alignment of entries before taking a difference. Matching lengths alone do not guarantee matching physical locations.

```python
U.shape == (Nx,Ny)
```

| expression | shape |  
| --- | --- |  
| `U` | `(Nx,Ny)` |  
| `U[1:-1,1:-1]` | `(Nx-2,Ny-2)` |  
| `U[2:,1:-1]` | `(Nx-2,Ny-2)` |  
| `U[:-2,1:-1]` | `(Nx-2,Ny-2)` |  
| `U[1:-1,2:]` | `(Nx-2,Ny-2)` |  
| `U[1:-1,:-2]` | `(Nx-2,Ny-2)` |

## 46. Reason about broadcast shapes

Write the aligned trailing axes explicitly. Distinguish coordinate axes from component axes even when their lengths coincide.

```python
X.shape = (Nx,1,1)
Y.shape = (1,Ny,1)
Z.shape = (1,1,Nz)
```

```python
X + Y + Z
```

```python
(Nx,Ny,Nz)
```

## 47. Questions for reading array code

Which points are selected, what shares storage, which axes broadcast, and which axes contract? These questions reveal the operation's structure.

## 48. Indexing map

Use basic slices for structured regions, masks for conditions, and advanced indexing for specified samples. They differ in shape, ordering, and copy behaviour.

| Numerical meaning | NumPy pattern |  
| --- | --- |  
| left boundary | `u[0]`, `U[0,:]` |  
| right boundary | `u[-1]`, `U[-1,:]` |  
| interior | `u[1:-1]`, `U[1:-1,1:-1]` |  
| left neighbor | `u[:-2]` |  
| center | `u[1:-1]` |  
| right neighbor | `u[2:]` |  
| x-face pair | `U[:-1,:]`, `U[1:,:]` |  
| y-face pair | `U[:,:-1]`, `U[:,1:]` |  
| region mask | boolean indexing |  
| component select | `V[...,0]` |  
| tensor component | `T[...,i,j]` |  
| add component axis | `a[...,None]` |  
| independent copy | `.copy()` |  
| automatic shape expansion | broadcasting |

## 49. Six ideas to retain

Track index origin, excluded stops, neighbour alignment, aliasing, broadcasting, and field location. Vectorization must preserve the intended update semantics.

## 50. Continue to P4

The next note distinguishes products and contractions through vectors, matrices, and tensors with explicit component conventions.

$$
\mathbf y=\mathbf A\mathbf x
$$

$$
\mathbf t=\boldsymbol\sigma\mathbf n
$$

$$
\sigma_n=\mathbf n\cdot\mathbf t
$$

$$
\tau_n=\mathbf m\cdot\mathbf t
$$

## Further work

1. Change one parameter or discretization choice and predict the effect before running.
2. Write the input and output shapes, units, and field locations.
3. Construct a deliberately invalid input and make the calculation report it clearly.
4. Separate an execution check from an accuracy check and from any physical claim.

## Primary documentation

- [NumPy indexing](https://numpy.org/doc/stable/user/basics.indexing.html)
- [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)

These are living documentation links. The accompanying requirements and verification report identify the versions actually executed for this edition.

## AI assistance and responsibility

The English adaptation, editorial supplementation, and code review were prepared with AI assistance. Wook Kang retains responsibility for reviewing the explanations, examples, and any subsequent research application. No new experimental validation is claimed.